# 05b Neural rankers: feature MLP and history attention

**Purpose.** The neural part of PROJECT_PLAN section 5 (item 4c), on a Colab GPU. Both models are compared with the final LambdaMART on the same validation searches:

1. **Feature MLP (E27, this part):** the same 19 inputs as the final LambdaMART (query length, relevance, lagged popularity, pooled history, item-item CF) with a ranking loss. The control: how much of LambdaMART's result is the features and how much the trees?
2. **History attention (next part):** the MLP plus attention over the shopper's last history events, weighted by the query and the candidate.

**Rules (as for LambdaMART, notebook 05).** Training on all clean training searches minus the same inner early-stopping set (3,757 dev searches); clamped searches and searches without an engaged candidate excluded; candidates were shuffled when the features were built; evaluation with our exact tie-aware evaluator on all validation lists, `validation_clean` as the main population; paired per-search differences. One fixed configuration per model, one debugging cycle, no tuning (PROJECT_PLAN 5, 28 Oct).

**Score with a popularity residual.** score = a x log(1 + lagged popularity) + f(inputs). The last layer of f starts at zero and a at 1, so the untrained model ranks exactly by lagged popularity, the deployable reference; training only has to learn what improves on it.

**Loss.** Listwise softmax cross-entropy over the complete list: the model's softmax over the candidates is pushed towards a target distribution proportional to the labels (0-3), so purchases count three times a click, as in our NDCG gains.

**How to run on Colab.** Copy to `MyDrive/BT4222 Group Project/processed/` (in addition to the v3 files from E21): `cf/cf1_train_dev.parquet`, `cf/cf1_train_rest.parquet`, `cf/cf1_validation.parquet` (05 section 10) and `neural/lambdamart_validation_scores.parquet` (05 section 14). Open this notebook from GitHub (branch `BRANCH` below), choose a GPU runtime and *Run all*. Results go to `processed/neural/`; download the executed notebook afterwards. Without a GPU the notebook runs a **smoke test** (2,000 training and 1,000 validation searches, one epoch) whose numbers are not results.

In [ ]:
# torch first: on Windows, importing numpy / pandas before torch can break torch's DLL loading.
import torch
import torch.nn as nn

from pathlib import Path
import json
import sys
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

BRANCH = "feat/models"  # branch to clone on Colab; "main" once merged
REPO_ROOT = next((p.resolve() for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "scripts/evaluation.py").is_file()), None)
if REPO_ROOT is None and IN_COLAB:  # opened from GitHub: clone the (public) repository
    import subprocess
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH,
                    "https://github.com/liamkunert/BT4222-group17-jdsearch.git", "/content/BT4222-group17-jdsearch"],
                   check=True)
    REPO_ROOT = Path("/content/BT4222-group17-jdsearch")
if REPO_ROOT is None:
    raise FileNotFoundError("Open/clone the repository checkout first.")
if IN_COLAB:
    drive.mount("/content/drive")
    PROCESSED = Path("/content/drive/MyDrive/BT4222 Group Project/processed")
else:
    PROCESSED = REPO_ROOT / "data/processed"
NEURAL_DIR = PROCESSED / "neural"
NEURAL_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO_ROOT))
from scripts.evaluation import evaluate, paired_difference, summarise

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SMOKE = DEVICE.type != "cuda"  # smoke test without a GPU: small subsets, one epoch
TAG = "_smoke" if SMOKE else ""
SEED, MODEL_SEED = 4222, 1
pd.options.display.float_format = "{:,.4f}".format
meta = json.loads((PROCESSED / "features_v3_meta.json").read_text())
searches = pd.read_parquet(PROCESSED / "search_table.parquet")
assignment = pd.read_csv(PROCESSED / "split_v1.csv", index_col="session_id")
print(f"device {DEVICE} ({torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU'}); smoke test: {SMOKE}; "
      f"torch {torch.__version__}; data {PROCESSED}")

## 1. Populations and inputs

The same training and early-stopping searches as LambdaMART in notebook 05 (reproduced here and checked by their counts), and the same 19 inputs as the final LambdaMART.

In [ ]:
dev_ids = assignment.index[(assignment.split == "train") & assignment.dev_subsample]
dev_info = searches.loc[dev_ids]
dev_pool = dev_ids[~(dev_info.final_gap_clamped | (dev_info.n_engaged == 0) | (dev_info.n_candidates == 1)).to_numpy()]
inner_ids = np.sort(np.random.default_rng(SEED).choice(dev_pool.to_numpy(), size=round(0.1 * len(dev_pool)), replace=False))
train_info = searches.loc[assignment.index[assignment.split == "train"]]
full_pool = train_info.index[(~train_info.final_gap_clamped & (train_info.n_engaged > 0)
                              & (train_info.n_candidates > 1)).to_numpy()]
fit_ids = np.setdiff1d(full_pool.to_numpy(), inner_ids)
assert (len(fit_ids), len(inner_ids)) == (121_989, 3_757)  # as in 05, section 11
validation_ids = assignment.index[assignment.split == "validation"].to_numpy()
if SMOKE:
    rng = np.random.default_rng(SEED)
    fit_ids = np.sort(rng.choice(fit_ids, 2_000, replace=False))
    inner_ids = np.sort(rng.choice(inner_ids, 500, replace=False))
    validation_ids = np.sort(rng.choice(validation_ids, 1_000, replace=False))
validation_sizes = searches.loc[validation_ids, "n_candidates"]

G = meta["groups"]
CF_COLUMNS = ["cf_sum", "cf_max", "cf_mean", "cf_count", "cf_history_in_graph"]
INPUTS = G["G0_search"] + G["G1_relevance"] + G["G2a_popularity_deployable"] + G["H1_pooled"] + CF_COLUMNS
POPULARITY = "pop_history_lag1d"
assert len(INPUTS) == 19


def load(split, sessions):
    """Session ids, wids, raw inputs (float32) and labels for the given searches of one split."""
    path = PROCESSED / f"features_v3_{split}.parquet"
    cf_path = PROCESSED / "cf" / f"cf1_{split}.parquet"
    ids = pq.read_table(path, columns=["session_id", "wid"]).to_pandas()
    cf_ids = pq.read_table(cf_path, columns=["session_id", "wid"]).to_pandas()
    assert ids.equals(cf_ids), f"CF rows of {split} do not line up with the v3 rows"
    keep = ids.session_id.isin(sessions).to_numpy()
    X = np.empty((int(keep.sum()), len(INPUTS)), dtype=np.float32)
    for j, column in enumerate(INPUTS):
        source = cf_path if column in CF_COLUMNS else path
        X[:, j] = pq.read_table(source, columns=[column]).column(0).to_numpy()[keep]
    labels = pq.read_table(path, columns=["label"]).column(0).to_numpy()[keep]
    return ids.session_id.to_numpy()[keep], ids.wid.to_numpy()[keep], X, labels


start = time.time()
parts = [load(split, fit_ids) for split in ("train_dev", "train_rest")]
fit = [np.concatenate([p[k] for p in parts]) for k in range(4)]
inner = load("train_dev", inner_ids)
valid = load("validation", validation_ids)
del parts
for name, (sid, _, X, y) in {"fit": fit, "inner": inner, "validation": valid}.items():
    print(f"{name}: {len(np.unique(sid)):,} searches, {len(sid):,} rows, engaged share {np.mean(y >= 1):.4f}")
print(f"loaded in {time.time() - start:.0f} s")

## 2. Input transformation

Fitted on the training rows only: inputs that are non-negative with a maximum above 10 (counts such as popularity, history matches, BM25) are log-transformed with log(1 + x); all inputs are standardised; missing values (unknown metadata, candidates outside the CF graph) become 0 after standardisation, with a 0/1 missing flag for every input that has missing values. Lagged popularity also enters the residual term as log(1 + x).

In [ ]:
class Inputs:
    """Log-transform, standardise and flag missing values; statistics from the training rows only."""

    def __init__(self, X):
        self.log = (np.nanmin(X, axis=0) >= 0) & (np.nanmax(X, axis=0) > 10)
        self.flag = np.isnan(X).any(axis=0)
        Z = self._log(X)
        self.mean, self.std = np.nanmean(Z, axis=0), np.nanstd(Z, axis=0) + 1e-6

    def _log(self, X):
        Z = X.copy()
        Z[:, self.log] = np.log1p(Z[:, self.log])
        return Z

    def __call__(self, X):
        Z = (self._log(X) - self.mean) / self.std
        missing = np.isnan(X[:, self.flag]).astype(np.float32)
        return np.hstack([np.nan_to_num(Z, nan=0.0), missing]).astype(np.float32)


transform = Inputs(fit[2])
pop_j = INPUTS.index(POPULARITY)
print(f"log-transformed: {[c for c, l in zip(INPUTS, transform.log) if l]}")
print(f"missing flags: {[c for c, f in zip(INPUTS, transform.flag) if f]}")


class Lists:
    """One split on the device: transformed inputs, residual term, labels and the row range of every search."""

    def __init__(self, data):
        sid, wid, X, y = data
        bounds = np.r_[0, np.flatnonzero(np.diff(sid)) + 1, len(sid)]
        assert len(bounds) - 1 == len(np.unique(sid)), "rows of a search must be contiguous"
        self.sid, self.wid, self.labels_np = sid, wid, y
        self.start, self.length = bounds[:-1], np.diff(bounds)
        self.x = torch.from_numpy(transform(X)).to(DEVICE)
        self.log_pop = torch.from_numpy(np.log1p(np.nan_to_num(X[:, pop_j], nan=0.0))).to(DEVICE)
        self.y = torch.from_numpy(y.astype(np.float32)).to(DEVICE)

    def __len__(self):
        return len(self.start)

    def batch(self, searches):
        """Padded index matrix (searches x longest list) and its mask for the given search positions."""
        start = torch.as_tensor(self.start[searches], device=DEVICE)
        length = torch.as_tensor(self.length[searches], device=DEVICE)
        offsets = torch.arange(int(length.max()), device=DEVICE)
        mask = offsets[None, :] < length[:, None]
        index = torch.where(mask, start[:, None] + offsets[None, :], torch.zeros_like(mask, dtype=torch.long))
        return index, mask


data = {"fit": Lists(fit), "inner": Lists(inner), "validation": Lists(valid)}
del fit, inner
print({name: len(lists) for name, lists in data.items()},
      f"inputs after transformation: {data['fit'].x.shape[1]}")

## 3. Model, loss and training loop

Shared by both neural models: a list is scored as a whole (padded to its longest member), the loss is the listwise softmax cross-entropy, and early stopping uses NDCG@10 with linear gains on the inner set (the final numbers come from our evaluator).

In [ ]:
class FeatureMLP(nn.Module):
    """score = a * log(1 + lagged popularity) + MLP(inputs); last layer at zero, a = 1: starts as popularity."""

    def __init__(self, n_inputs, hidden=(128, 64), dropout=0.1):
        super().__init__()
        layers, width = [], n_inputs
        for h in hidden:
            layers += [nn.Linear(width, h), nn.ReLU(), nn.Dropout(dropout)]
            width = h
        self.body = nn.Sequential(*layers)
        self.out = nn.Linear(width, 1)
        nn.init.zeros_(self.out.weight)
        nn.init.zeros_(self.out.bias)
        self.alpha = nn.Parameter(torch.tensor(1.0))

    def forward(self, lists, index, mask):
        x, log_pop = lists.x[index], lists.log_pop[index]
        return self.alpha * log_pop + self.out(self.body(x)).squeeze(-1)


def listwise_loss(scores, labels, mask):
    """Softmax cross-entropy against targets proportional to the labels (lists with an engaged candidate only)."""
    scores = scores.masked_fill(~mask, -1e9)
    target = labels * mask
    target = target / target.sum(dim=1, keepdim=True).clamp(min=1e-9)
    return -(target * torch.log_softmax(scores, dim=1)).sum(dim=1).mean()


DISCOUNT = 1 / torch.log2(torch.arange(2, 12, device=DEVICE, dtype=torch.float32))


def batch_ndcg(scores, labels, mask, k=10):
    """NDCG@k with linear gains per list (ties in sort order; for early stopping only)."""
    scores = scores.masked_fill(~mask, -float("inf"))
    labels = labels * mask
    top = torch.topk(scores, min(k, scores.shape[1]), dim=1).indices
    dcg = (labels.gather(1, top) * DISCOUNT[:top.shape[1]]).sum(dim=1)
    ideal = (torch.topk(labels, top.shape[1], dim=1).values * DISCOUNT[:top.shape[1]]).sum(dim=1)
    keep = ideal > 0
    return (dcg[keep] / ideal[keep]).sum().item(), int(keep.sum())


@torch.no_grad()
def predict(model, lists, batch_size=512):
    """Scores for every row of a split, in row order."""
    model.eval()
    scores = torch.empty(len(lists.y), device=DEVICE)
    for lo in range(0, len(lists), batch_size):
        index, mask = lists.batch(np.arange(lo, min(lo + batch_size, len(lists))))
        scores[index[mask]] = model(lists, index, mask)[mask]
    return scores


@torch.no_grad()
def inner_ndcg(model, batch_size=512):
    model.eval()
    lists, total, n = data["inner"], 0.0, 0
    for lo in range(0, len(lists), batch_size):
        index, mask = lists.batch(np.arange(lo, min(lo + batch_size, len(lists))))
        s, m = batch_ndcg(model(lists, index, mask), lists.y[index], mask)
        total, n = total + s, n + m
    return total / n


def train(model, epochs=10, patience=2, batch_size=256, lr=1e-3, label=""):
    """Adam on the fit searches, early stopping on the inner set's NDCG@10; returns the best model and a log."""
    torch.manual_seed(MODEL_SEED)
    rng = np.random.default_rng(MODEL_SEED)
    model.to(DEVICE)
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    history = [{"epoch": 0, "loss": np.nan, "inner_ndcg_at_10": inner_ndcg(model), "seconds": 0.0}]
    best, best_state, waited = history[0]["inner_ndcg_at_10"], {k: v.clone() for k, v in model.state_dict().items()}, 0
    lists = data["fit"]
    for epoch in range(1, (1 if SMOKE else epochs) + 1):
        model.train()
        start, losses = time.time(), []
        for batch in np.array_split(rng.permutation(len(lists)), max(1, len(lists) // batch_size)):
            index, mask = lists.batch(batch)
            loss = listwise_loss(model(lists, index, mask), lists.y[index], mask)
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
            losses.append(loss.item())
        score = inner_ndcg(model)
        history.append({"epoch": epoch, "loss": float(np.mean(losses)), "inner_ndcg_at_10": score,
                        "seconds": round(time.time() - start, 1)})
        print(f"{label} epoch {epoch}: loss {np.mean(losses):.4f}, inner NDCG@10 {score:.4f}, {time.time() - start:.0f} s")
        if score > best:
            best, best_state, waited = score, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            waited += 1
            if waited >= patience:
                break
    model.load_state_dict(best_state)
    return model, pd.DataFrame(history).set_index("epoch")

Check before training: the untrained model must rank exactly by lagged popularity (same scores up to the constant factor a = 1).

In [ ]:
torch.manual_seed(MODEL_SEED)
untrained = FeatureMLP(data["fit"].x.shape[1]).to(DEVICE)
check = predict(untrained, data["validation"]).cpu().numpy()
assert np.allclose(check, np.log1p(np.nan_to_num(valid[2][:, pop_j], nan=0.0)), atol=1e-5)
print("untrained model = lagged popularity: ok")

## 4. E27: feature MLP

In [ ]:
torch.manual_seed(MODEL_SEED)
start = time.time()
mlp, mlp_history = train(FeatureMLP(data["fit"].x.shape[1]), label="MLP")
mlp_seconds = time.time() - start
mlp_scores = predict(mlp, data["validation"]).cpu().numpy()
print(f"trained in {mlp_seconds / 60:.1f} min; a = {mlp.alpha.item():.3f}")
mlp_history

## 5. Evaluation against the references

All validation lists with our evaluator; `validation_clean` is the main population. References on the same searches: lagged popularity, and the LambdaMART base, M1 and M1 + CF trained on all training searches (scores exported from notebook 05, section 14).

In [ ]:
sid, wid, X_valid, y_valid = valid
frame = pd.DataFrame({"session_id": sid, "wid": wid, "label": y_valid})
reference_file = NEURAL_DIR / "lambdamart_validation_scores.parquet"
reference = pd.read_parquet(reference_file) if reference_file.is_file() else None
rankings = {"feature MLP": mlp_scores, "lagged history popularity": np.nan_to_num(X_valid[:, pop_j], nan=0.0)}
if reference is not None:  # rows in v3 order, like `valid`
    reference = reference[reference.session_id.isin(validation_ids).to_numpy()]
    assert (reference.session_id.to_numpy() == sid).all() and (reference.wid.to_numpy() == wid).all()
    rankings.update({"LambdaMART base": reference["base"].to_numpy(), "LambdaMART M1": reference["m1"].to_numpy(),
                     "LambdaMART M1 + CF": reference["m1_cf"].to_numpy()})
per_search = {name: evaluate(frame.assign(score=s), "score", expected_sizes=validation_sizes)
              for name, s in rankings.items()}
all_ids = per_search["feature MLP"].index
clean_ids = all_ids[~searches.final_gap_clamped.reindex(all_ids).to_numpy()]
rows = {}
for name in rankings:
    row = summarise(per_search[name].loc[clean_ids]).iloc[0].to_dict()
    row["ndcg_all_searches"] = per_search[name].ndcg.mean()
    if name != "feature MLP":
        d = paired_difference(per_search["feature MLP"].loc[clean_ids], per_search[name].loc[clean_ids])
        row.update(mlp_minus_this=d[0], low=d[1], high=d[2])
    rows[name] = row
results = pd.DataFrame(rows).T
results["searches"] = results.searches.astype(int)
results

In [ ]:
pd.DataFrame({"session_id": sid, "wid": wid, "score": mlp_scores}).to_parquet(
    NEURAL_DIR / f"mlp_validation_scores{TAG}.parquet", index=False)
summary = {"model": "feature MLP", "inputs": INPUTS, "transformed_inputs": int(data["fit"].x.shape[1]),
           "fit_searches": len(data["fit"]), "inner_searches": len(data["inner"]), "smoke": SMOKE,
           "device": str(DEVICE), "minutes": round(mlp_seconds / 60, 1), "alpha": float(mlp.alpha.item()),
           "best_epoch": int(mlp_history.inner_ndcg_at_10.idxmax()),
           "ndcg_clean": float(per_search["feature MLP"].ndcg.loc[clean_ids].mean())}
(NEURAL_DIR / f"mlp_summary{TAG}.json").write_text(json.dumps(summary, indent=2))
summary